# Test Your Understanding of Tensor Dimensions — SOLUTIONS

This notebook contains the completed answers for every `...` and `#TODO` block.

In [ ]:
import torch

---
## Part 1: Shape and Rank

In [ ]:
a = torch.tensor(7)
b = torch.tensor([1, 2, 3, 4])
c = torch.tensor([[1, 2], [3, 4], [5, 6]])
d = torch.randn(2, 3, 4)

# Answers:
#   a.shape = ()       a.ndim = 0    a.numel() = 1
#   b.shape = (4,)     b.ndim = 1    b.numel() = 4
#   c.shape = (3, 2)   c.ndim = 2    c.numel() = 6
#   d.shape = (2,3,4)  d.ndim = 3    d.numel() = 24

for name, t in [('a', a), ('b', b), ('c', c), ('d', d)]:
    print(f"{name}: shape={t.shape}, ndim={t.ndim}, numel={t.numel()}")

---
## Part 2: Understanding `dim` in Reductions

In [ ]:
x = torch.tensor([[1, 2, 3],
                   [4, 5, 6],
                   [7, 8, 9]])

print("x:")
print(x)
print(f"x.shape = {x.shape}")
print()

# sum(dim=0) collapses rows → sums each column: shape (3,)
print("sum(dim=0):", x.sum(dim=0))    # tensor([12, 15, 18])
# sum(dim=1) collapses columns → sums each row: shape (3,)
print("sum(dim=1):", x.sum(dim=1))    # tensor([ 6, 15, 24])
# dim=-1 is the last dim = dim 1
print("sum(dim=-1):", x.sum(dim=-1))  # tensor([ 6, 15, 24])

### Exercise 2a: 3-D Reductions

In [ ]:
x = torch.arange(24).reshape(2, 3, 4)
print("x.shape =", x.shape)
print(x)
print()

# dim=0 collapses the first axis (size 2) → shape (3, 4)
s0 = x.sum(dim=0)
# dim=1 collapses the second axis (size 3) → shape (2, 4)
s1 = x.sum(dim=1)
# dim=2 collapses the third axis (size 4) → shape (2, 3)
s2 = x.sum(dim=2)

print(f"sum(dim=0): shape={s0.shape}")
print(s0)
print()
print(f"sum(dim=1): shape={s1.shape}")
print(s1)
print()
print(f"sum(dim=2): shape={s2.shape}")
print(s2)

### Exercise 2b: Your Turn

In [1]:
images = torch.tensor([
    [[1, 2, 3], [4, 5, 6], [7, 8, 9]],
    [[9, 8, 7], [6, 5, 4], [3, 2, 1]],
    [[1, 1, 1], [1, 1, 1], [1, 1, 1]],
    [[0, 0, 0], [0, 9, 0], [0, 0, 0]],
], dtype=torch.float)

print("images.shape =", images.shape)  # (4, 3, 3)

# Mean per image: reduce over dims 1 and 2 (the H and W of each image)
mean_per_image = images.mean(dim=(1, 2))
print("Mean per image:", mean_per_image)  # tensor([5., 5., 1., 1.])

# Pixel-wise max across the batch: reduce over dim 0
max_across_batch = images.max(dim=0).values
print("Max across batch:\n", max_across_batch)

NameError: name 'torch' is not defined

---
## Part 3: `keepdim` and Broadcasting

In [ ]:
x = torch.tensor([[1.0, 2.0, 3.0],
                   [4.0, 5.0, 6.0]])

# Without keepdim → shape (2,)
row_sum = x.sum(dim=1)
print(f"row_sum shape (no keepdim): {row_sum.shape}")  # (2,)

# With keepdim → shape (2, 1)
row_sum_kd = x.sum(dim=1, keepdim=True)
print(f"row_sum shape (keepdim):    {row_sum_kd.shape}")  # (2, 1)

# x / row_sum → (2, 3) / (2,) → broadcasts row_sum as if shape (1, 2)
#   which tries (2, 3) / (1, 2) → MISMATCH on dim 1 (3 vs 2) → ERROR!
# x / row_sum_kd → (2, 3) / (2, 1) → broadcasts correctly → (2, 3)
normalized = x / row_sum_kd
print("Normalized rows:\n", normalized)
print("Row sums (should be 1.0):", normalized.sum(dim=1))

### Exercise 3a: Manual Softmax

In [ ]:
logits = torch.tensor([[2.0, 1.0, 0.1],
                        [0.5, 2.5, 1.0]])

# softmax(x_i) = exp(x_i) / sum(exp(x_j))
exp_logits = torch.exp(logits)                        # element-wise exp
my_softmax = exp_logits / exp_logits.sum(dim=1, keepdim=True)  # normalize each row

expected = torch.softmax(logits, dim=1)
print("Your softmax:\n", my_softmax)
print("Expected:\n", expected)
print("Match:", torch.allclose(my_softmax, expected))

---
## Part 4: Squeeze and Unsqueeze

In [ ]:
v = torch.tensor([10, 20, 30])  # shape: (3,)

# Answers:
print("unsqueeze(0):", v.unsqueeze(0).shape)   # (1, 3)
print("unsqueeze(1):", v.unsqueeze(1).shape)   # (3, 1)
print("unsqueeze(-1):", v.unsqueeze(-1).shape) # (3, 1)
print("[None, :]:", v[None, :].shape)          # (1, 3)
print("[:, None]:", v[:, None].shape)           # (3, 1)

In [ ]:
x = torch.randn(1, 3, 1, 5, 1)

# Answers:
print("original:", x.shape)             # (1, 3, 1, 5, 1)
print("squeeze():", x.squeeze().shape)   # (3, 5)         — all size-1 dims removed
print("squeeze(0):", x.squeeze(0).shape) # (3, 1, 5, 1)   — only dim 0 removed
print("squeeze(1):", x.squeeze(1).shape) # (1, 3, 1, 5, 1) — dim 1 is size 3, NOT removed
print("squeeze(2):", x.squeeze(2).shape) # (1, 3, 5, 1)   — only dim 2 removed

### Exercise 4a: Fix the Shape

In [ ]:
single_image = torch.randn(3, 32, 32)  # shape: (C, H, W)

# Add batch dimension at position 0
batched_image = single_image.unsqueeze(0)
print("batched_image.shape =", batched_image.shape)  # (1, 3, 32, 32)
assert batched_image.shape == torch.Size([1, 3, 32, 32])

# Remove batch dimension
model_output = torch.randn(1, 10)
prediction = model_output.squeeze(0)
print("prediction.shape =", prediction.shape)  # (10,)
assert prediction.shape == torch.Size([10])

---
## Part 5: Reshape, View, Flatten

In [ ]:
x = torch.arange(24)
print("x.shape =", x.shape)  # (24,)

a = x.reshape(6, 4)       # shape: (6, 4)
b = x.reshape(2, 3, 4)    # shape: (2, 3, 4)
c = x.reshape(4, -1)      # shape: (4, 6)  — PyTorch infers 24/4 = 6

print(f"a.shape = {a.shape}")  # (6, 4)
print(f"b.shape = {b.shape}")  # (2, 3, 4)
print(f"c.shape = {c.shape}")  # (4, 6)

In [ ]:
images = torch.randn(16, 3, 28, 28)

# Flatten from dim 1 onward: collapses C, H, W into one dim
flat = images.flatten(start_dim=1)
print(f"flat.shape = {flat.shape}")  # (16, 2352)
assert flat.shape == torch.Size([16, 2352])

# Flatten from dim 2 onward: collapses only H, W
spatial_flat = images.flatten(start_dim=2)
print(f"spatial_flat.shape = {spatial_flat.shape}")  # (16, 3, 784)
assert spatial_flat.shape == torch.Size([16, 3, 784])

---
## Part 6: Transpose and Permute

In [ ]:
x = torch.randn(2, 3, 4)

# Answers:
print("transpose(0,1):", x.transpose(0, 1).shape)   # (3, 2, 4)
print("transpose(1,2):", x.transpose(1, 2).shape)   # (2, 4, 3)
print("permute(2,0,1):", x.permute(2, 0, 1).shape)  # (4, 2, 3)

### Exercise 6a: Channel Conversion

In [ ]:
images_nhwc = torch.randn(8, 32, 32, 3)  # (N, H, W, C)

# NHWC → NCHW: move dim 3 (C) to position 1
images_nchw = images_nhwc.permute(0, 3, 1, 2)
print(f"images_nchw.shape = {images_nchw.shape}")  # (8, 3, 32, 32)
assert images_nchw.shape == torch.Size([8, 3, 32, 32])

# NCHW → NHWC: move dim 1 (C) to position 3
images_back = images_nchw.permute(0, 2, 3, 1)
print(f"images_back.shape = {images_back.shape}")  # (8, 32, 32, 3)
assert images_back.shape == torch.Size([8, 32, 32, 3])

---
## Part 7: Cat, Stack, Unbind

In [ ]:
a = torch.ones(2, 3)
b = torch.zeros(2, 3)

# Answers:
print("cat dim=0:", torch.cat([a, b], dim=0).shape)     # (4, 3)
print("cat dim=1:", torch.cat([a, b], dim=1).shape)     # (2, 6)
print("stack dim=0:", torch.stack([a, b], dim=0).shape) # (2, 2, 3)  — new dim inserted
print("stack dim=1:", torch.stack([a, b], dim=1).shape) # (2, 2, 3)  — new dim at pos 1

### Exercise 7a: Build a Batch

In [ ]:
samples = [torch.randn(3, 16, 16) for _ in range(5)]

# stack creates a new dimension (the batch dim) at position 0
batch = torch.stack(samples, dim=0)
print(f"batch.shape = {batch.shape}")  # (5, 3, 16, 16)
assert batch.shape == torch.Size([5, 3, 16, 16])

# unbind removes dim 0 and returns a tuple of tensors
individuals = torch.unbind(batch, dim=0)
print(f"Number of images: {len(individuals)}")        # 5
print(f"Each image shape: {individuals[0].shape}")    # (3, 16, 16)
assert len(individuals) == 5
assert individuals[0].shape == torch.Size([3, 16, 16])

---
## Part 8: Gather and Scatter

In [ ]:
logits = torch.tensor([[0.1, 0.9, 0.2, 0.3, 0.5],
                        [0.8, 0.1, 0.3, 0.6, 0.2],
                        [0.2, 0.3, 0.7, 0.1, 0.4],
                        [0.1, 0.2, 0.1, 0.9, 0.3]])

labels = torch.tensor([1, 0, 2, 3])

# gather needs index to have the same number of dims as input.
# labels is (4,), we need it to be (4, 1) so gather picks one value per row.
# Then squeeze the result from (4, 1) to (4,).
correct_logits = torch.gather(logits, dim=1, index=labels.unsqueeze(1)).squeeze(1)
print("Logits at correct classes:", correct_logits)  # tensor([0.9, 0.8, 0.7, 0.9])

In [ ]:
labels = torch.tensor([0, 2, 1, 4, 3])
num_classes = 5

# Start with zeros, scatter 1s into the correct column positions.
# labels must be 2-D for scatter_: shape (5, 1)
one_hot = torch.zeros(labels.size(0), num_classes, dtype=torch.long)
one_hot.scatter_(dim=1, index=labels.unsqueeze(1), value=1)
print("One-hot:\n", one_hot)
# tensor([[1, 0, 0, 0, 0],
#         [0, 0, 1, 0, 0],
#         [0, 1, 0, 0, 0],
#         [0, 0, 0, 0, 1],
#         [0, 0, 0, 1, 0]])

---
## Part 9: Expand and Repeat

In [ ]:
x = torch.tensor([[1], [2], [3]])  # shape: (3, 1)

# expand(3, 4) → (3, 4): each row's single value is repeated across 4 columns
print("expand(3,4):\n", x.expand(3, 4))
# tensor([[1, 1, 1, 1],
#         [2, 2, 2, 2],
#         [3, 3, 3, 3]])

# -1 means "keep the existing size" → same as expand(3, 4)
print("expand(-1,4):\n", x.expand(-1, 4))
print()

y = torch.tensor([1, 2, 3])  # shape: (3,)
# repeat(3) tiles the tensor 3 times: shape (9,)
print("repeat(3):", y.repeat(3))           # tensor([1, 2, 3, 1, 2, 3, 1, 2, 3])
# repeat(2, 3) first adds a dim → (1, 3), then tiles 2x in dim 0, 3x in dim 1: shape (2, 9)
print("repeat(2,3):\n", y.repeat(2, 3))    # 2 rows, each is [1,2,3,1,2,3,1,2,3]

### Exercise 9a: Broadcasting vs Expand

In [ ]:
data = torch.tensor([[10.0, 20.0, 30.0],
                      [40.0, 50.0, 60.0]])

row_means = data.mean(dim=1, keepdim=True)  # shape: (2, 1)
centered_v1 = data - row_means.expand(-1, 3)

# Broadcasting handles (2, 3) - (2, 1) automatically — no expand needed
centered_v2 = data - row_means

print("Centered (expand):\n", centered_v1)
print("Centered (broadcast):\n", centered_v2)
print("Same result:", torch.allclose(centered_v1, centered_v2))

---
## Part 10: Flip, Roll, and Narrow

In [ ]:
x = torch.tensor([[1, 2, 3],
                   [4, 5, 6],
                   [7, 8, 9]])

# flip dims=[0]: reverse row order
print("flip dims=[0]:\n", torch.flip(x, dims=[0]))
# [[7, 8, 9],
#  [4, 5, 6],
#  [1, 2, 3]]

# flip dims=[1]: reverse column order in each row
print("flip dims=[1]:\n", torch.flip(x, dims=[1]))
# [[3, 2, 1],
#  [6, 5, 4],
#  [9, 8, 7]]

# flip both: reverse rows AND columns (180° rotation)
print("flip dims=[0,1]:\n", torch.flip(x, dims=[0, 1]))
# [[9, 8, 7],
#  [6, 5, 4],
#  [3, 2, 1]]

In [ ]:
x = torch.tensor([1, 2, 3, 4, 5])

# roll +2: shift right by 2, elements that fall off the right wrap to the left
print("roll 2:", torch.roll(x, shifts=2))     # tensor([4, 5, 1, 2, 3])
# roll -1: shift left by 1, element that falls off the left wraps to the right
print("roll -1:", torch.roll(x, shifts=-1))   # tensor([2, 3, 4, 5, 1])

### Exercise 10a: Crop an Image

In [ ]:
image = torch.arange(48).reshape(3, 4, 4)  # (C=3, H=4, W=4)
print("Full image shape:", image.shape)
print("Channel 0:\n", image[0])

# narrow(dim, start, length): select a slice along a dim
# First narrow H (dim=1) to rows 0..1, then narrow W (dim=2) to cols 0..1
crop = image.narrow(1, 0, 2).narrow(2, 0, 2)
print(f"\ncrop.shape = {crop.shape}")  # (3, 2, 2)
print("Channel 0 of crop:\n", crop[0])
# tensor([[0, 1],
#         [4, 5]])
assert crop.shape == torch.Size([3, 2, 2])

---
## Part 11: Contiguity

In [ ]:
x = torch.arange(12).reshape(3, 4)
print("x is contiguous:", x.is_contiguous())  # True

y = x.transpose(0, 1)  # (4, 3)
print("y is contiguous:", y.is_contiguous())  # False

# y.view(12) raises RuntimeError because y is not contiguous
# Fix 1: make it contiguous first, then view
flat_fix1 = y.contiguous().view(12)
# Fix 2: reshape handles non-contiguous tensors automatically
flat_fix2 = y.reshape(12)

print("Fix 1 (.contiguous().view()):", flat_fix1)
print("Fix 2 (.reshape()):           ", flat_fix2)

---
## Part 12: Putting It All Together

In [ ]:
batch_size, num_heads, seq_len = 2, 4, 8
attn_scores = torch.randn(batch_size, num_heads, seq_len, seq_len)
print(f"Input: {attn_scores.shape}")  # (2, 4, 8, 8)

# Step 1: softmax over last dim (dim=-1)
attn_probs = torch.softmax(attn_scores, dim=-1)
print(f"After softmax: {attn_probs.shape}")  # (2, 4, 8, 8)

row_sums = attn_probs.sum(dim=-1)
print(f"Row sums (should all be ~1.0): min={row_sums.min():.4f}, max={row_sums.max():.4f}")

# Step 2: average across heads (dim=1 is the heads dim)
avg_attn = attn_probs.mean(dim=1)
print(f"Averaged over heads: {avg_attn.shape}")  # (2, 8, 8)
assert avg_attn.shape == torch.Size([2, 8, 8])

# Step 3: flatten the 8x8 maps into vectors (flatten from dim=1)
flat_attn = avg_attn.flatten(start_dim=1)
print(f"Flattened: {flat_attn.shape}")  # (2, 64)
assert flat_attn.shape == torch.Size([2, 64])

# Step 4: argmax along last dim of avg_attn to find most-attended position per query
most_attended = avg_attn.argmax(dim=-1)
print(f"Most attended positions: {most_attended.shape}")  # (2, 8)
print(most_attended)
assert most_attended.shape == torch.Size([2, 8])

In [ ]:
print("All done! If you got here without errors, you understand tensor dimensions well.")